In [145]:
%pip install pandas
%pip install wordninja
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import wordninja
import os
import string


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [146]:
#load dataset
df = pd.read_csv('equities_data.csv')
print(df.head())


                             title  \
0                            Stock   
1                    Thor Equities   
2                Sterling Equities   
3                             TIAA   
4  Alexandria Real Estate Equities   

                                             summary  \
0  \nStocks(alsocapital stock, or sometimes inter...   
1  Thor Equitiesis a principal investment firm sp...   
2  Sterling Equitiesis a diversified, family-run ...   
3  TheTeachers Insurance and Annuity Association ...   
4  Alexandria Real Estate Equities, Inc.is areal ...   

                                             content  \
0  \nPublic market\nExchange·Securities\nBond val...   
1  Thor Equitiesis a principal investment firm sp...   
2  Sterling Equitiesis a diversified, family-run ...   
3  TheTeachers Insurance and Annuity Association ...   
4  Alexandria Real Estate Equities, Inc.is areal ...   

                                               links  \
0  ['/wiki/Private_equity_secondary_marke

In [147]:
# checking for missing values
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")

title
False    2185
Name: count, dtype: int64

summary
False    1858
True      327
Name: count, dtype: int64

content
False    1861
True      324
Name: count, dtype: int64

links
False    2185
Name: count, dtype: int64

url
False    2185
Name: count, dtype: int64



In [148]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

number of duplicate rows:  (64, 5)


In [149]:
#dropping duplicates
df = df.drop_duplicates()
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
2180    False
2181    False
2182    False
2183    False
2184    False
Length: 2121, dtype: bool


In [150]:
# Drop rows where both columns 'content' and 'summary' are empty
df1 = df.copy()
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [151]:
#checking for missing values after removing duplicates
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    1808
Name: count, dtype: int64

summary
False    1805
True        3
Name: count, dtype: int64

content
False    1808
Name: count, dtype: int64

links
False    1808
Name: count, dtype: int64

url
False    1808
Name: count, dtype: int64



In [152]:
# having only 3 rows with missing we can replace them with Not Available

df1 = df.copy()

df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

df_cleaned.loc[df_cleaned['summary'].isna(), 'summary'] = df_cleaned.loc[df_cleaned['summary'].isna(), 'content'].apply(get_summary)


C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\3346600660.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.loc[df_cleaned['summary'].isna(), 'summary'] = df_cleaned.loc[df_cleaned['summary'].isna(), 'content'].apply(get_summary)


In [153]:
# checking if there are any missing values left
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    1808
Name: count, dtype: int64

summary
False    1808
Name: count, dtype: int64

content
False    1808
Name: count, dtype: int64

links
False    1808
Name: count, dtype: int64

url
False    1808
Name: count, dtype: int64



In [154]:
#removing \n from the content table and summary
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\1462990387.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\1462990387.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)


In [155]:
df_cleaned.head()


,title,summary,content,links,url
0,Stock,"Stocks(alsocapital stock, or sometimes interc...",Public market Exchange·Securities Bond valuat...,"['/wiki/Private_equity_secondary_market', '/wi...",https://en.wikipedia.org/wiki/Stock
1,Thor Equities,Thor Equitiesis a principal investment firm sp...,Thor Equitiesis a principal investment firm sp...,"['/wiki/Palmer_House_Hilton', '/wiki/Martha_St...",https://en.wikipedia.org/wiki/Thor_Equities
2,Sterling Equities,"Sterling Equitiesis a diversified, family-run ...","Sterling Equitiesis a diversified, family-run ...","['/wiki/Nelson_Doubleday_Jr.', '/wiki/Great_Ne...",https://en.wikipedia.org/wiki/Sterling_Equities
3,TIAA,TheTeachers Insurance and Annuity Association ...,TheTeachers Insurance and Annuity Association ...,"['/wiki/Tiaa_(disambiguation)', '/wiki/For-pro...",https://en.wikipedia.org/wiki/TIAA
4,Alexandria Real Estate Equities,"Alexandria Real Estate Equities, Inc.is areal ...","Alexandria Real Estate Equities, Inc.is areal ...","['/wiki/Seattle', '/wiki/Pasadena,_California'...",https://en.wikipedia.org/wiki/Alexandria_Real_...


In [156]:
# Apply word splitting
df_cleaned['summary'] =df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)))
df_cleaned['content'] =df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)))

C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\667218373.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['summary'] =df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)))
C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\667218373.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['content'] =df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)))


In [157]:
df_cleaned['links'] = df['links']

df_cleaned.head()

C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\29228091.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['links'] = df['links']


,title,summary,content,links,url
0,Stock,Stocks also capital stock or sometimes interch...,Public market Exchange Securities Bond valuati...,"['/wiki/Private_equity_secondary_market', '/wi...",https://en.wikipedia.org/wiki/Stock
1,Thor Equities,Thor Equities is a principal investment firm s...,Thor Equities is a principal investment firm s...,"['/wiki/Palmer_House_Hilton', '/wiki/Martha_St...",https://en.wikipedia.org/wiki/Thor_Equities
2,Sterling Equities,Sterling Equities is a diversified family run ...,Sterling Equities is a diversified family run ...,"['/wiki/Nelson_Doubleday_Jr.', '/wiki/Great_Ne...",https://en.wikipedia.org/wiki/Sterling_Equities
3,TIAA,The Teachers Insurance and Annuity Association...,The Teachers Insurance and Annuity Association...,"['/wiki/Tiaa_(disambiguation)', '/wiki/For-pro...",https://en.wikipedia.org/wiki/TIAA
4,Alexandria Real Estate Equities,Alexandria Real Estate Equities Inc is a real ...,Alexandria Real Estate Equities Inc is a real ...,"['/wiki/Seattle', '/wiki/Pasadena,_California'...",https://en.wikipedia.org/wiki/Alexandria_Real_...


In [158]:
# removing links from the summary, content and title columns


def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))

df_cleaned = df_cleaned.drop(columns=['links'])

C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\1118466056.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\1118466056.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
C:\Users\Lish Ai Labs\AppData\Local\Temp\ipykernel_11104\1118466056.py:13: SettingWithCopyWarning: 
A value is tryin

In [159]:
df_cleaned.tail(10)

,title,summary,content,url
2175,KYKN,KY KN 1430 AM is a commercial radio station li...,Kei z er Oregon Compass Media Networks Premier...,https://en.wikipedia.org/wiki/KYKN
2176,List of Arizona ballot propositions,The following is a partial list of Arizona bal...,1912 1916 1920 1924 1928 1932 1936 1940 1944 1...,https://en.wikipedia.org/wiki/List_of_Arizona_...
2177,National Intelligence Assessments on Infectiou...,The United States Intelligence Community IC ha...,The United States Intelligence Community IC ha...,https://en.wikipedia.org/wiki/National_Intelli...
2178,Ron Carter (businessman),Sir Ronald Powell Carter ON Z KN ZM born 17 Ju...,Sir Ronald Powell Carter ON Z KN ZM born 17 Ju...,https://en.wikipedia.org/wiki/Ron_Carter_(busi...
2179,Joshua Green (businessman),Joshua Green October 16 1869 January 24 1975 2...,Joshua Green October 16 1869 January 24 1975 w...,https://en.wikipedia.org/wiki/Joshua_Green_(bu...
2180,2010 in baseball,The following are the baseball events of the y...,The following are the baseball events of the y...,https://en.wikipedia.org/wiki/2010_in_baseball
2181,2009 in baseball,The following are the baseball events of the y...,The following are the baseball events of the y...,https://en.wikipedia.org/wiki/2009_in_baseball
2182,"List of public art in Washington, D.C., Ward 6",This is a list of public art inWard 6 of Washi...,Download coordinates as KM L G PX all coordina...,https://en.wikipedia.org/wiki/List_of_public_a...
2183,2011 in baseball,The following are the baseball events of the y...,The following are the baseball events of the y...,https://en.wikipedia.org/wiki/2011_in_baseball
2184,La Roue Tourangelle,La Rou e Tour angel le is a road bicycle race ...,La Rou e Tour angel le is a road bicycle race ...,https://en.wikipedia.org/wiki/La_Roue_Tourangelle


In [160]:
#resetting the index
df_cleaned = df_cleaned.reset_index(drop=True)  # drop=True removes old index

In [161]:
# Normalize text
text_cols = ['content', 'links', 'summary', 'title', 'url']
for col in text_cols:
  df[col] = df[col].str.lower()

In [162]:
# Tokenize the dataset
text_cols = ['content', 'links', 'summary', 'title', 'url']

#Apply tokenization
for col in text_cols:
  df[f'{col}_tokens'] = df[col].astype(str).apply(lambda x: x.split())
print (df.head())

                             title  \
0                            stock   
1                    thor equities   
2                sterling equities   
3                             tiaa   
4  alexandria real estate equities   

                                             summary  \
0  \nstocks(alsocapital stock, or sometimes inter...   
1  thor equitiesis a principal investment firm sp...   
2  sterling equitiesis a diversified, family-run ...   
3  theteachers insurance and annuity association ...   
4  alexandria real estate equities, inc.is areal ...   

                                             content  \
0  \npublic market\nexchange·securities\nbond val...   
1  thor equitiesis a principal investment firm sp...   
2  sterling equitiesis a diversified, family-run ...   
3  theteachers insurance and annuity association ...   
4  alexandria real estate equities, inc.is areal ...   

                                               links  \
0  ['/wiki/private_equity_secondary_marke

In [163]:
# saving the cleaned dataset
 
folder_path = os.path.join(os.path.expanduser("~"), "Desktop", "Simba") 

file_name = "cleaned_equities.csv"
file_path = os.path.join(folder_path, file_name)

df.to_csv(file_path, index=False, encoding='utf-8')

print(f"Cleaned data saved to: {file_path}")

Cleaned data saved to: C:\Users\Lish Ai Labs\Desktop\Simba\cleaned_equities.csv
